# First Checkpoint Notebook

This notebook documents the **first checkpoint** crawled data during our crawling process. For this stage, our primary objective is to verify the feasibility of the crawling approach on GitHub.

This notebook is executed on Google Colab Pro+ because it allows the code to keep running even if the we turn off our computer or exit the tab.

Colab Link: https://colab.research.google.com/drive/1gKFD2ojfZ6KkZ5xwt_bjXVCxdwlM2IQC?usp=sharing

## 1. Initial Data Collection Phase

During the early stage of the crawling process, our team focused on:
- Testing crawler stability and request handling
- Validating data extraction logic
- Ensuring collected data aligns with the requirements of an **implicit feedback–based recommendation system**

Due to the exploratory nature of this phase, the crawling process was executed multiple times with incremental adjustments to the crawler configuration and parsing rules.

## 2. Logging File Limitation

One drawback encountered when executed code on colab is that, the log file was not successfully preserved. We had ran into some issues when maintaining a log file on the Colab Environment, these issues are:
- Temporary execution environments
- Frequent crawler restarts during debugging

Our team had decided not to mount Google Drive since

- Google Drive enforces storage and I/O quota limits, which may interrupt long-running crawling jobs
- Mounting Drive introduces additional I/O overhead that can negatively affect crawling stability
- To prioritize uninterrupted crawling execution, local temporary storage was used instead

In [ ]:
import os, csv, time, json, requests
from datetime import datetime
from collections import defaultdict, deque

# ─────────────────────────────────────────────
# (Mục tiêu 1.2M+ Interactions)
# ─────────────────────────────────────────────
GITHUB_TOKEN = "PERSONAL_ACCESS_TOKEN"

BASE_URL = "https://api.github.com"

# Targets sau k-core
FINAL_USERS_MIN  = 10_000
FINAL_INT_MIN    = 1_200_000

# Crawl buffer
CRAWL_USERS_TARGET = 35_000
CRAWL_INT_TARGET   = 3_500_000

# K-core params
MIN_INTERACTIONS = 5
MIN_USER_PER_ITEM = 3

# API page limits
MAX_STARRED_PAGES = 30
MAX_SEARCH_PAGES  = 10

# Dirs (Nên dùng đường dẫn Google Drive nếu chạy trên Colab)
OUT_DIR      = "output_github_v3"
RAW_DIR      = os.path.join(OUT_DIR, "raw")
LIGHTGCN_DIR = os.path.join(OUT_DIR, "lightgcn")
ULTRAGCN_DIR = os.path.join(OUT_DIR, "ultragcn")
IMREC_DIR    = os.path.join(OUT_DIR, "imrec")
PROFILES_DIR = os.path.join(OUT_DIR, "profiles")
CHECKPOINT   = os.path.join(OUT_DIR, "checkpoint.json")
RAW_CSV      = os.path.join(RAW_DIR, "interactions.csv")
RAW_REPO_CSV = os.path.join(RAW_DIR, "repos.csv")

# Tạo tất cả thư mục ngay từ đầu
for d in [OUT_DIR, RAW_DIR, LIGHTGCN_DIR, ULTRAGCN_DIR, IMREC_DIR, PROFILES_DIR]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────
# SMART QUERY GENERATOR
# ─────────────────────────────────────────────
def generate_smart_queries():
    """Tạo query chia nhỏ dải follower để lấy tối đa user"""
    langs = ["python", "javascript", "java", "go", "rust", "typescript", "cpp", "php", "ruby", "swift"]
    # Chia nhỏ dải followers: 50-60, 61-70...
    ranges = [
        (50, 65), (66, 80), (81, 95), (96, 110), (111, 130),
        (131, 155), (156, 185), (186, 220), (221, 260), (261, 310),
        (311, 400), (401, 600), (601, 1000), (1001, 5000)
    ]
    queries = []
    for lang in langs:
        for low, high in ranges:
            queries.append(f"followers:{low}..{high} language:{lang}")
    return queries

# ─────────────────────────────────────────────
# REQUEST HELPERS (Thay logging bằng print)
# ─────────────────────────────────────────────

def _handle_rate_limit(r, is_search=False):
    remaining = int(r.headers.get("X-RateLimit-Remaining", 999))
    reset_at  = int(r.headers.get("X-RateLimit-Reset", time.time() + 60))

    if r.status_code in [403, 429]:
        wait = max(reset_at - int(time.time()), 30) + 5
        print(f"\n[!] Rate limit! Ngủ {wait}s...")
        time.sleep(wait)
        return False

    if r.status_code == 200:
        if is_search:
            time.sleep(2.2) # Search API giới hạn 30 req/min
            if remaining < 5:
                wait = max(reset_at - int(time.time()), 10) + 3
                time.sleep(wait)
        else:
            if remaining < 100:
                wait = max(reset_at - int(time.time()), 10) + 2
                time.sleep(wait)
    return True

def _get_rest(url, params=None):
    for attempt in range(5):
        try:
            r = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3.star+json"}, params=params, timeout=15)
            if not _handle_rate_limit(r, is_search=False): continue
            return r.json() if r.status_code == 200 else []
        except Exception as e:
            time.sleep(5)
    return []

def _get_search(url, params=None):
    for attempt in range(5):
        try:
            r = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"}, params=params, timeout=15)
            if not _handle_rate_limit(r, is_search=True): continue
            return r.json() if r.status_code == 200 else {}
        except Exception as e:
            time.sleep(5)
    return {}

# ─────────────────────────────────────────────
# CHECKPOINT
# ─────────────────────────────────────────────
def save_cp(data: dict):
    with open(CHECKPOINT, "w") as f:
        json.dump(data, f)

def load_cp() -> dict:
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT) as f:
            return json.load(f)
    return {}

# ─────────────────────────────────────────────
# PHASE 1: Collect users (Smart Strategy)
# ─────────────────────────────────────────────
def collect_users(cp: dict):
    if cp.get("phase") in ["crawl", "process", "done"]:
        return cp["users"]

    users = list(cp.get("users", []))
    done_queries = set(cp.get("done_queries", []))
    visited = set(users)
    SEARCH_QUERIES = generate_smart_queries()

    print(f"--- PHASE 1: Tìm {CRAWL_USERS_TARGET} User chất lượng ---")
    for query in SEARCH_QUERIES:
        if query in done_queries or len(users) >= CRAWL_USERS_TARGET: continue

        for page in range(1, MAX_SEARCH_PAGES + 1):
            if len(users) >= CRAWL_USERS_TARGET: break
            data = _get_search(f"{BASE_URL}/search/users", {"q": query, "per_page": 100, "page": page, "sort": "repositories"})
            items = data.get("items", []) if isinstance(data, dict) else []
            if not items: break

            for u in items:
                login = u.get("login")
                if login and login not in visited:
                    visited.add(login); users.append(login)

            print(f"\r   Query: {query[:35]}... | Đã lấy: {len(users):,}", end="")
            if len(items) < 100: break

        done_queries.add(query)
        save_cp({"phase": "collect", "users": users, "done_queries": list(done_queries)})

    print(f"\nDONE Phase 1: {len(users):,} users.")
    save_cp({"phase": "crawl", "users": users, "crawl_idx": 0, "total_interactions": 0})
    return users

# ─────────────────────────────────────────────
# PHASE 2: Crawl stars (Deep Star)
# ─────────────────────────────────────────────
def crawl(users, cp: dict):
    start_idx = cp.get("crawl_idx", 0)
    total_int = cp.get("total_interactions", 0)
    if start_idx >= len(users): return RAW_CSV, RAW_REPO_CSV

    print(f"--- PHASE 2: Crawl Star (Mục tiêu: {CRAWL_INT_TARGET:,}) ---")
    seen_repos = set()
    mode = "a" if start_idx > 0 else "w"

    with open(RAW_CSV, mode, newline="", encoding="utf-8") as f_int, \
         open(RAW_REPO_CSV, mode, newline="", encoding="utf-8") as f_repo:
        w_int, w_repo = csv.writer(f_int), csv.writer(f_repo)
        if start_idx == 0:
            w_int.writerow(["user", "repo", "timestamp"])
            w_repo.writerow(["repo", "language", "stars", "description"])

        for idx in range(start_idx, len(users)):
            u = users[idx]
            user_ints = []

            # Lấy sâu 30 trang
            for page in range(1, MAX_STARRED_PAGES + 1):
                data = _get_rest(f"{BASE_URL}/users/{u}/starred", {"per_page": 100, "page": page})
                if not data or not isinstance(data, list): break

                for item in data:
                    repo = item["repo"]
                    name = repo.get("full_name")
                    ts = int(datetime.strptime(item["starred_at"], "%Y-%m-%dT%H:%M:%SZ").timestamp())
                    user_ints.append((u, name, ts))
                    if name not in seen_repos:
                        seen_repos.add(name)
                        w_repo.writerow([name, repo.get("language") or "N/A", repo.get("stargazers_count", 0), ""])
                if len(data) < 100: break

            if user_ints:
                w_int.writerows(user_ints)
                total_int += len(user_ints)

            if idx % 20 == 0:
                print(f"\r   User {idx}/{len(users)} | Tổng tương tác: {total_int:,}", end="")

            if idx % 200 == 0:
                save_cp({"phase": "crawl", "users": users, "crawl_idx": idx + 1, "total_interactions": total_int})

            if total_int >= CRAWL_INT_TARGET:
                print("\n[!] Đã đạt mục tiêu crawl thô.")
                break

    save_cp({"phase": "process", "users": users, "total_interactions": total_int})
    return RAW_CSV, RAW_REPO_CSV



def main():
    print("="*50 + "\nGITHUB CRAWLER V3 - TARGET 1.2M\n" + "="*50)
    cp = load_cp()
    users = collect_users(cp)
    raw_int, raw_repo = crawl(users, load_cp())



if __name__ == "__main__":
    main()

GITHUB CRAWLER V3 - TARGET 1.2M
--- PHASE 1: Tìm 35000 User chất lượng ---
   Query: followers:156..185 language:java... | Đã lấy: 35,000
DONE Phase 1: 35,000 users.
--- PHASE 2: Crawl Star (Mục tiêu: 3,500,000) ---
   User 6620/35000 | Tổng tương tác: 3,382,310

# Result

During code execution, we encountered an **unexpected runtime interruption** while running the notebook on **Google Colab Pro+**, which caused the execution to terminate before all code cells could be fully completed.

Such interruptions may occur due to platform-level constraints, including:
- Session time limits for long-running processes
- Resource reallocation or preemption on shared infrastructure
- Intensive network and I/O operations during crawling

As a result, the entire pipeline could not be executed in a single uninterrupted session.

Fortunately, prior to the runtime disconnection, we successfully **downloaded and preserved the intermediate output data** generated up to that point.  
The collected datasets remain valid and are sufficient for subsequent preprocessing and analysis steps.

After that, we crawled the remaining data to reach more than 1 million interactions and more than 6000 users locally

